In [20]:
import json 
import os
import requests
from datetime import datetime, timedelta
import sys
import csv

In [2]:
current_notebook_directory = os.getcwd() 
project_root = os.path.abspath(os.path.join(current_notebook_directory, '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Config.LoadMetaData import metadata

In [3]:
def get_data(api_url):
    try:
        response = requests.get(
            api_url,
            headers={'Content-Type': 'application/json'}
        )
        
        response.raise_for_status()

        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
        return None

def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")

def fetch_and_concatenate_data(apis):
    all_data = []
    for api_url in apis:
        data = get_data(api_url)
        if data:  
            all_data.extend(data)
        else:
            print(f"Failed to fetch data from {api_url}")
    return all_data

def load_json_from_file(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None
    
    try:
        with open(file_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
        return data
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {file_path}. The file might be corrupted or not in valid JSON format.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while reading {file_path}: {e}")
        return None

In [4]:
def calculate_guest_counts(reservations):
    """
    Calculates the total number of adults, children, and infants from a list of reservations.
    """
    if not isinstance(reservations, list) or len(reservations) == 0:
        return {"adults": 0, "children": 0, "infants": 0, "kids": 0, "total": 0}
    
    total_adults = sum(reservation.get("adult", 0) or 0 for reservation in reservations)
    total_children = sum(reservation.get("child", 0) or 0 for reservation in reservations)
    total_infants = sum(reservation.get("infant", 0) or 0 for reservation in reservations)
    
    total_kids = total_children + total_infants
    total_guests = total_adults + total_kids
    
    return {
        "adults": total_adults,
        "children": total_children,
        "infants": total_infants,
        "kids": total_kids,
        "total": total_guests
    }

def count_guest_nationalities(reservations):
    """
    Counts the occurrences of each guest nationality from a list of reservations.
    
    Args:
        reservations: List of reservation dictionaries
    """
    nationalities_count = {}
    
    for reservation in reservations:
        nationality = reservation.get("guestNationality")
        if nationality:  
            if nationality in nationalities_count:
                nationalities_count[nationality] += 1
            else:
                nationalities_count[nationality] = 1
    
    return nationalities_count


def count_customer_countries(reservations):
    """
    Counts the occurrences of each customer country from a list of reservations.
    
    Args:
        reservations: List of reservation dictionaries
    """
    countries_count = {}
    
    for reservation in reservations:
        country = reservation.get("customerCountry")
        if country:  
            if country in countries_count:
                countries_count[country] += 1
            else:
                countries_count[country] = 1
    
    return countries_count

def calculate_total_revenue(reservations):
    """
    Calculates the total revenue from the localCurrencyPrice field for a list of reservations.
    
    Args:
        reservations: List of reservation dictionaries
    """
    if not reservations:
        return 0.0
    
    total_revenue = 0.0
    
    for reservation in reservations:
        price = reservation.get("localCurrencyPrice", 0)
        if price is None:
            price = 0
        
        total_revenue += price
    
    return total_revenue


def get_avr_of_sejour(actives, client=None):
    """
    Calculates the average length of stay for all reservations or for a specific client.
    """
    if not actives:
        return 0
        
    if client:
        reservations_to_use = [reservation for reservation in actives if reservation.get("customer") == client["name"]]
    else:
        reservations_to_use = actives
    
    if not reservations_to_use:
        return 0
        
    total_nights = sum(reservation.get("nights", 0) or 0 for reservation in reservations_to_use)
    return total_nights / len(reservations_to_use)


def convert_date_fr_to_iso(date_fr):
    """
    Converts a date from French format (DD-MM-YYYY) to ISO format (YYYY-MM-DD).
    """
    if isinstance(date_fr, str) and len(date_fr.split("-")) == 3:
        day, month, year = date_fr.split("-")
        return f"{year}-{month}-{day}"
    return date_fr


def compter_reservations_actives_par_dates(reservations, dates, total_rooms, client=None):
    """
    Counts active reservations by date and calculates various metrics.
    """
    
    lignes = [
        'Total rooms', 'Room nights', 'Occupancy percentage', 'Bed nights',
        'adults', 'kids', 'Available rooms', 'Revenue', 'IF',
        'ADR Nte', 'ADR (Average daily rate)', 'Revpar'
    ]
    
    resultats = {ligne: {} for ligne in lignes}
    line_totals = {ligne: 0 for ligne in lignes}
    
    for date_donnee in dates:
        date_cible = datetime.fromisoformat(date_donnee).date()
        
        reservations_actives = []
        for reservation in reservations:
            date_debut = datetime.fromisoformat(convert_date_fr_to_iso(reservation["checkin"])).date()
            date_fin = datetime.fromisoformat(convert_date_fr_to_iso(reservation["checkout"])).date()
            
            if client:
                if date_cible >= date_debut and date_cible < date_fin and reservation.get("customer") == client["name"]:
                    reservations_actives.append(reservation)
            else:
                if date_cible >= date_debut and date_cible < date_fin:
                    reservations_actives.append(reservation)
        
        for ligne in lignes:
            resultats[ligne][date_donnee] = calcul(reservations_actives, ligne, total_rooms, line_totals)
    
    return resultats, line_totals


def calcul(reservations_actives, ligne, total_rooms, line_totals):
    """
    Calculates a specific metric for a list of active reservations.
    """
    if ligne == 'Total rooms':
        line_totals[ligne] += total_rooms
        return total_rooms
        
    elif ligne == 'Room nights':
        room_nights = len(reservations_actives)
        line_totals[ligne] += room_nights
        return room_nights
        
    elif ligne == 'Occupancy percentage':
        room_nights = len(reservations_actives)
        percentage = (room_nights / total_rooms) * 100 if total_rooms > 0 else 0
        line_totals[ligne] = (line_totals['Room nights'] / line_totals['Total rooms']) * 100 if line_totals['Total rooms'] > 0 else 0
        return round(percentage, 2)
        
    elif ligne == 'Bed nights':
        bed_nights = sum((res.get("adult", 0) or 0) + (res.get("child", 0) or 0) + (res.get("infant", 0) or 0) 
                         for res in reservations_actives)
        line_totals[ligne] += bed_nights
        return bed_nights
        
    elif ligne == 'adults':
        adults = sum(res.get("adult", 0) or 0 for res in reservations_actives)
        line_totals[ligne] += adults
        return adults
        
    elif ligne == 'kids':
        kids = sum((res.get("child", 0) or 0) + (res.get("infant", 0) or 0) for res in reservations_actives)
        line_totals[ligne] += kids
        return kids
        
    elif ligne == 'Available rooms':
        available = total_rooms - len(reservations_actives)
        line_totals[ligne] = line_totals['Total rooms'] - line_totals['Room nights']
        return available
        
    elif ligne == 'Revenue':
        revenue = sum((res.get("localCurrencyPrice", 0) or 0) / (res.get("nights", 1) or 1) for res in reservations_actives)
        line_totals[ligne] += revenue
        return round(revenue, 2)
        
    elif ligne == 'IF':
        bed_nights = sum((res.get("adult", 0) or 0) + (res.get("child", 0) or 0) + (res.get("infant", 0) or 0) 
                         for res in reservations_actives)
        room_nights = len(reservations_actives)
        if_value = bed_nights / room_nights if room_nights > 0 else 0
        line_totals[ligne] = line_totals['Bed nights'] / line_totals['Room nights'] if line_totals['Room nights'] > 0 else 0
        return round(if_value, 2)
        
    elif ligne == 'ADR Nte':
        bed_nights = sum((res.get("adult", 0) or 0) + (res.get("child", 0) or 0) + (res.get("infant", 0) or 0) 
                         for res in reservations_actives)
        revenue = sum((res.get("localCurrencyPrice", 0) or 0) / (res.get("nights", 1) or 1) for res in reservations_actives)
        adr_nte = revenue / bed_nights if bed_nights > 0 else 0
        line_totals[ligne] = line_totals['Revenue'] / line_totals['Bed nights'] if line_totals['Bed nights'] > 0 else 0
        return round(adr_nte, 2)
        
    elif ligne == 'ADR (Average daily rate)':
        room_nights = len(reservations_actives)
        revenue = sum((res.get("localCurrencyPrice", 0) or 0) / (res.get("nights", 1) or 1) for res in reservations_actives)
        adr = revenue / room_nights if room_nights > 0 else 0
        line_totals[ligne] = line_totals['Revenue'] / line_totals['Room nights'] if line_totals['Room nights'] > 0 else 0
        return round(adr, 2)
        
    elif ligne == 'Revpar':
        revenue = sum((res.get("localCurrencyPrice", 0) or 0) / (res.get("nights", 1) or 1) for res in reservations_actives)
        revpar = revenue / total_rooms if total_rooms > 0 else 0
        line_totals[ligne] = line_totals['Revenue'] / line_totals['Total rooms'] if line_totals['Total rooms'] > 0 else 0
        return round(revpar, 2)
    
    return 0

def get_dates(from_date, to_date):
    from datetime import datetime, timedelta
    
    start_date = datetime.strptime(from_date, "%Y-%m-%d")
    end_date = datetime.strptime(to_date, "%Y-%m-%d")
    
    dates = []
    current_date = start_date
    
    while current_date <= end_date:
        dates.append(current_date.strftime("%Y-%m-%d"))
        current_date += timedelta(days=1)
    
    return dates

def get_occupancy_data(start_date_str, end_date_str, total_rooms_param, api_templates=None):
    """
    Fetches reservation data for a given date range, calculates occupancy,
    and returns the occupancy data per date and line totals.
    """

    apis_for_date_range = [template.format(start_date_str, end_date_str) for template in api_templates]
    print(f"Sending requests to the following APIs: {len(apis_for_date_range)}")

    reservation_data = fetch_and_concatenate_data(apis_for_date_range)

    if not reservation_data:
        print("No data fetched. Cannot calculate occupancy.")
        return None
    
    print(f"Successfully fetched {len(reservation_data)} reservations.")

    dates_in_range = get_dates(start_date_str, end_date_str)
    print(f"Calculating occupancy for dates: {len(dates_in_range)}")
    
    occupancy_results, line_totals_results = compter_reservations_actives_par_dates(reservation_data, dates_in_range, total_rooms_param)
    print("Occupancy calculation complete.")
    
    return {
        "occupancy_par_date": occupancy_results,
        "line_totals": line_totals_results
    }

In [5]:
metadata['PMSInformation']

{'api_templates': ['https://pmsvaleriaapi.fractalstay.com/api/reservations?&from={}&to={}&group=1&resastatus=0',
  'https://pmsvaleriaapi.fractalstay.com/api/reservations?&from={}&to={}&group=1&resastatus=3',
  'https://pmsvaleriaapi.fractalstay.com/api/reservations?&from={}&to={}&group=1&resastatus=2'],
 'start_date_range': '2024-01-01',
 'end_date_range': '2025-12-30',
 'output_path': '../Data/PMSEtractedData/PMSreservations.json',
 'final_data_path': '../Data/TransformedPMSData/final_reservation_metrics.json'}

In [6]:
data = load_json_from_file(metadata['PMSInformation']['output_path'])

In [7]:
def group_reservations_by_dates(reservations, start_date_str, end_date_str):
    """
    Groups reservations by date for all dates between start_date and end_date.
    For each date, includes all reservations where the date falls between checkin and checkout.
    
    Args:
        reservations: List of reservation dictionaries
        start_date_str: Start date in ISO format "YYYY-MM-DD"
        end_date_str: End date in ISO format "YYYY-MM-DD"
    """
    dates = get_dates(start_date_str, end_date_str)
    
    result = {date: [] for date in dates}
    
    for reservation in reservations:
        checkin_iso = convert_date_fr_to_iso(reservation["checkin"])
        checkout_iso = convert_date_fr_to_iso(reservation["checkout"])
        
        checkin_date = datetime.fromisoformat(checkin_iso).date()
        checkout_date = datetime.fromisoformat(checkout_iso).date()
        
        for date_str in dates:
            date_obj = datetime.fromisoformat(date_str).date()
            
            if checkin_date <= date_obj < checkout_date:
                result[date_str].append(reservation)
    
    return result

In [8]:
dataFT = group_reservations_by_dates(reservations=data, 
                                     start_date_str=metadata['PMSInformation']['start_date_range'],
                                     end_date_str=metadata['PMSInformation']['end_date_range'])

In [9]:
append_to_json_file(dataFT, '../Data/TransformedPMSData/reservations_by_dates.json')

Data saved to ../Data/TransformedPMSData/reservations_by_dates.json


In [10]:
def count_room_types_by_date(grouped_reservations):
    """
    For each date in the grouped reservations, counts the occurrences of each room type.
    
    Args:
        grouped_reservations: Dictionary with dates as keys and lists of reservations as values
                             (output from group_reservations_by_dates function)
    """
    room_types_by_date = {}
    
    for date, reservations in grouped_reservations.items():
        room_types_count = {}
        
        for reservation in reservations:
            room_type = reservation.get("roomType", "Unknown")
            
            if room_type in room_types_count:
                room_types_count[room_type] += 1
            else:
                room_types_count[room_type] = 1
        
        room_types_by_date[date] = room_types_count
    
    return room_types_by_date

In [11]:
count_room_types_by_date(dataFT)

{'2024-01-01': {},
 '2024-01-02': {},
 '2024-01-03': {},
 '2024-01-04': {},
 '2024-01-05': {},
 '2024-01-06': {},
 '2024-01-07': {},
 '2024-01-08': {},
 '2024-01-09': {},
 '2024-01-10': {},
 '2024-01-11': {},
 '2024-01-12': {},
 '2024-01-13': {},
 '2024-01-14': {},
 '2024-01-15': {},
 '2024-01-16': {},
 '2024-01-17': {},
 '2024-01-18': {},
 '2024-01-19': {},
 '2024-01-20': {},
 '2024-01-21': {},
 '2024-01-22': {},
 '2024-01-23': {},
 '2024-01-24': {},
 '2024-01-25': {},
 '2024-01-26': {},
 '2024-01-27': {},
 '2024-01-28': {},
 '2024-01-29': {},
 '2024-01-30': {},
 '2024-01-31': {},
 '2024-02-01': {},
 '2024-02-02': {},
 '2024-02-03': {},
 '2024-02-04': {},
 '2024-02-05': {},
 '2024-02-06': {},
 '2024-02-07': {},
 '2024-02-08': {},
 '2024-02-09': {},
 '2024-02-10': {},
 '2024-02-11': {},
 '2024-02-12': {},
 '2024-02-13': {},
 '2024-02-14': {},
 '2024-02-15': {},
 '2024-02-16': {},
 '2024-02-17': {},
 '2024-02-18': {},
 '2024-02-19': {},
 '2024-02-20': {},
 '2024-02-21': {},
 '2024-02-22

In [12]:
def classify_room_types(grouped_reservations):
    """
    Groups reservations by date and then classifies them into 'Standard' or 'Family' categories.
    Skips processing for dates with no reservations.
    """
    classified_data = {}
    
    for date, reservations in grouped_reservations.items():
        if not reservations:
            classified_data[date] = {
                "Standard": [],
                "Family": []
            }
            continue
        
        classified_data[date] = {
            "Standard": [],
            "Family": []
        }
        
        for reservation in reservations:
            room_type = reservation.get("roomType", "") or ""
            room_type = room_type.lower()
            
            if "family" in room_type:
                classified_data[date]["Family"].append(reservation)
            elif "standard" in room_type:
                classified_data[date]["Standard"].append(reservation)
            else:
                classified_data[date]["Standard"].append(reservation)
    
    return classified_data

In [13]:
dataGrouped = classify_room_types(dataFT)

In [14]:
append_to_json_file(dataGrouped, '../Data/TransformedPMSData/reservations_grouped_by_roomType.json')

Data saved to ../Data/TransformedPMSData/reservations_grouped_by_roomType.json


In [15]:
def calculate_metrics_for_reservations(reservations_actives, total_rooms):
    """
    Calculates various hospitality metrics for a list of active reservations.
    
    Args:
        reservations_actives: List of reservation dictionaries active on the given date
        total_rooms: Total number of rooms available
    """
    metrics = {}
    
    # Total rooms
    metrics['Total rooms'] = total_rooms
    
    # Room nights
    room_nights = len(reservations_actives)
    metrics['Room nights'] = room_nights
    
    # Occupancy percentage
    occupancy = (room_nights / total_rooms) * 100 if total_rooms > 0 else 0
    metrics['Occupancy percentage'] = round(occupancy, 2)
    
    # Bed nights (sum of all guests)
    bed_nights = sum((res.get("adult", 0) or 0) + (res.get("child", 0) or 0) + (res.get("infant", 0) or 0) 
                     for res in reservations_actives)
    metrics['Bed nights'] = bed_nights
    
    # Adults count
    adults = sum(res.get("adult", 0) or 0 for res in reservations_actives)
    metrics['adults'] = adults
    
    # Kids count (children + infants)
    kids = sum((res.get("child", 0) or 0) + (res.get("infant", 0) or 0) for res in reservations_actives)
    metrics['kids'] = kids
    
    # Available rooms
    metrics['Available rooms'] = total_rooms - room_nights
    
    # Revenue (daily)
    revenue = sum((res.get("localCurrencyPrice", 0) or 0) / (res.get("nights", 1) or 1) for res in reservations_actives)
    metrics['Revenue'] = round(revenue, 2)
    
    # IF (Bed nights / Room nights)
    if_value = bed_nights / room_nights if room_nights > 0 else 0
    metrics['IF'] = round(if_value, 2)
    
    # ADR Nte (Revenue / Bed nights)
    adr_nte = revenue / bed_nights if bed_nights > 0 else 0
    metrics['ADR Nte'] = round(adr_nte, 2)
    
    # ADR (Revenue / Room nights)
    adr = revenue / room_nights if room_nights > 0 else 0
    metrics['ADR (Average daily rate)'] = round(adr, 2)
    
    # Revpar (Revenue / Total rooms)
    revpar = revenue / total_rooms if total_rooms > 0 else 0
    metrics['Revpar'] = round(revpar, 2)
    
    return metrics


In [16]:
calculate_metrics_for_reservations(dataGrouped['2024-06-09']['Standard'], 432)

{'Total rooms': 432,
 'Room nights': 0,
 'Occupancy percentage': 0.0,
 'Bed nights': 0,
 'adults': 0,
 'kids': 0,
 'Available rooms': 432,
 'Revenue': 0,
 'IF': 0,
 'ADR Nte': 0,
 'ADR (Average daily rate)': 0,
 'Revpar': 0.0}

In [17]:
def calculate_all_metrics_by_date(grouped_data, standard_rooms=432, family_rooms=106):
    """
    Calculates metrics for each date and room type in the grouped data structure.
    
    Args:
        grouped_data: Dictionary with dates and room types as keys
                     {'date': {'Standard': [...], 'Family': [...]}, ...}
        standard_rooms: Total number of standard rooms available
        family_rooms: Total number of family rooms available
    """
    metrics_by_date = {}
    
    for date, room_types in grouped_data.items():
        metrics_by_date[date] = {}
        
        standard_metrics = calculate_metrics_for_reservations(room_types['Standard'], standard_rooms)
        
        if standard_metrics['Occupancy percentage'] > 100:
            standard_metrics['Occupancy percentage'] = 100.0
            
        standard_metrics['Total Revenue'] = calculate_total_revenue(room_types['Standard'])
        standard_metrics['Guest Nationalities'] = count_guest_nationalities(room_types['Standard'])
        standard_metrics['Customer Countries'] = count_customer_countries(room_types['Standard'])
        
        metrics_by_date[date]['Standard'] = standard_metrics
        
        family_metrics = calculate_metrics_for_reservations(room_types['Family'], family_rooms)
        
        if family_metrics['Occupancy percentage'] > 100:
            family_metrics['Occupancy percentage'] = 100.0
            
        family_metrics['Total Revenue'] = calculate_total_revenue(room_types['Family'])
        family_metrics['Guest Nationalities'] = count_guest_nationalities(room_types['Family'])
        family_metrics['Customer Countries'] = count_customer_countries(room_types['Family'])
        
        metrics_by_date[date]['Family'] = family_metrics
    
    return metrics_by_date

In [18]:
metricData = calculate_all_metrics_by_date(dataGrouped, standard_rooms=432, family_rooms=432)

In [19]:
append_to_json_file(metricData, '../Data/TransformedPMSData/final_reservation_metrics.json')

Data saved to ../Data/TransformedPMSData/final_reservation_metrics.json


In [21]:
def convert_json_to_csv(json_path, csv_output_path):
    """
    Convertit un fichier JSON contenant les données des chambres Standard (166) et Family (167)
    en un fichier CSV avec les colonnes : date, total_price_166, taux_occupation_166, total_price_167, taux_occupation_167
    """

    with open(json_path, "r") as f:
        data = json.load(f)

    rows = []

    for date, types in data.items():
        standard = types.get("Standard", {})
        family = types.get("Family", {})

        row = {
            "date": date,
            "total_price_166": standard.get("Total Revenue", 0),
            "taux_occupation_166": standard.get("Occupancy percentage", 0),
            "total_price_167": family.get("Total Revenue", 0),
            "taux_occupation_167": family.get("Occupancy percentage", 0),
        }
        rows.append(row)

    output_dir = os.path.dirname(csv_output_path)
    os.makedirs(output_dir, exist_ok=True)

    with open(csv_output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "date", 
            "total_price_166", 
            "taux_occupation_166", 
            "total_price_167", 
            "taux_occupation_167"
        ])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Fichier CSV généré avec succès : {csv_output_path}")

In [22]:
convert_json_to_csv(
    json_path = "../Data/TransformedPMSData/final_reservation_metrics.json",
    csv_output_path = "../Data/TransformedPMSData/finalCSVPMS.csv"
)

Fichier CSV généré avec succès : ../Data/TransformedPMSData/finalCSVPMS.csv
